# 5. 하네스 코어

반복해서 사용한 루프, 도구 표, 오류 처리, 예산, 트레이스를 한 블록으로 모은다.
이후 회차에서 복제하므로 코어는 40줄 이하로 유지한다.

### 공통 설정

`1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [1]:
import json
import os
import shutil
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 다섯 부분

| 부분 | 책임 | 이것이 없으면 |
|---|---|---|
| Runtime | 루프를 돌리고 언제 멈출지 정한다 | 무한히 돌거나 예산을 넘긴다 |
| ToolRegistry | 도구의 이름, 스키마, 실행, 오류를 든다 | 모델이 무엇을 할 수 있는지 모른다 |
| Session | 무엇이 컨텍스트에 들어가는지 정한다 | 컨텍스트가 자라다 넘친다 |
| Trace | 무슨 일이 있었는지 남긴다 (실제 서비스에서는 배열이 아닌 elk 같은 로거를 둔다) | 실패를 고칠 수 없다 |
| Policy | 무엇을 실행해도 되는지 검사한다 (승인 필요, 금지 행위) | 되돌릴 수 없는 일이 그냥 일어난다 |

프롬프트, 도구 설명, 예산, 오류 문구처럼 실행 중에는 고정되고 실험에서 바뀌는 값은 `CONFIG`로 분리한다.

### 도구는 하네스가 아니다

도구는 회차마다 바뀌고 하네스는 바뀌지 않는다. 그 경계가 보이도록 도구를 코어 블록 밖에 둔다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def list_docs():
    return [f.stem for f in sorted(CORPUS.glob("*.md"))]


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


def write_note(name, text):
    Path("work").mkdir(exist_ok=True)
    (Path("work") / name).write_text(text, encoding="utf-8")
    return f"{name} 에 {len(text)}자"

모델이 보는 도구 표와 이름을 함수에 잇는 `IMPL`이다.

In [3]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("list_docs", "문서 제목을 모두 돌려준다.", {}),
         spec("grep_docs", "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
              {"pattern": {"type": "string", "description": "찾을 낱말"}}),
         spec("write_note", "메모를 파일로 저장한다.",
              {"name": {"type": "string", "description": "파일 이름"},
               "text": {"type": "string", "description": "내용"}})]

IMPL = {"list_docs": list_docs, "grep_docs": grep_docs, "write_note": write_note}

### 코어: 두 함수와 CONFIG

코어는 두 셀 아래에 있는 `call_tool`과 `run`이다. 도구가 바뀌어도, 지시문이나 예산이 바뀌어도 이 두 함수는 그대로 쓴다.
그 앞에 두는 `CONFIG`는 코드가 아니라 데이터다. 코어가 읽는 키(지시문, 예산 셋, 승인이 필요한 도구, 오류 문구 둘)는 코어와 함께 고정하고, 값은 실험과 회차에 따라 바꾼다.
그래서 코어 함수 안에는 바뀔 값을 직접 적지 않는다.

In [ ]:
# 전체에 대한 선택
CONFIG = {
    "instructions": "도구로 확인한 것만 답한다. 세지 않은 숫자는 쓰지 않는다.",     # 시스템 프롬프트
    "max_turns": 6, "max_tokens": 60000, "max_seconds": 90,     # 예산
    "gated": ["write_note"],    # 사용자의 승인이 필요한 도구
    "gate_error": "error: 이 도구는 승인이 필요하다. 무엇을 할지 사용자에게 먼저 확인한다.",    # gated에서 approve가 되지 않은 경우
    "tool_error": "error: {e}. 인자를 고쳐 다시 부른다.",   # tool에 대한 오류 발생 시 에러메시지를 넣는 값
}

`call_tool`이 ToolRegistry와 Policy, `run`이 Runtime, Session, Trace다. 주석이 다섯 부분의 자리를 표시한다.

In [4]:
# 툴을 호출하는 함수
def call_tool(name, args, approve):             # ToolRegistry
    if name in CONFIG["gated"] and not approve:  # Policy
        return CONFIG["gate_error"]
    try:
        return json.dumps(IMPL[name](**args), ensure_ascii=False)
    except Exception as e:
        return CONFIG["tool_error"].format(e=e) # 인자를 고쳐서 다시 부르게끔

# ReACT, plan 을 쓰지 않은 5개 요소를 다 넣은 run 함수
def run(task, approve=False):                   # Runtime
    messages = [{"role": "user", "content": task}]          # Session
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})            # Trace
        if not calls:   # 실행이 정상적으로 완료
            return r.output_text, "완료", trace
        if used > CONFIG["max_tokens"]: # 토큰 예산 문제로 끝
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > CONFIG["max_seconds"]:    # 시간 예산 문제로 끝
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), approve)}
                     for c in calls]
    return None, "반복 예산", trace     # 턴 예산 문제로 끝

### 코어 길이 제한

열여섯 회차에서 복제하며 한 화면 안에서 구조를 파악할 수 있도록 빈 줄을 뺀 코어를 40줄 이하로 제한한다.
한계를 넘으면 하네스가 아니라 프레임워크가 된 것이다. 그때는 나눌 것이 아니라 뺄 것을 찾는다.

In [7]:
# 핵심적인 하네스 코어 (가장 기본 형태)
import inspect

core = inspect.getsource(call_tool) + inspect.getsource(run)
lines = [l for l in core.splitlines() if l.strip()]
print(f"코어 {len(lines)}줄 (빈 줄 제외)   CONFIG {len(CONFIG)}개 항목")

코어 28줄 (빈 줄 제외)   CONFIG 7개 항목


### 돌려 본다

정지 사유와 답이 따로 돌아온다.

In [8]:
answer, why, trace = run("한국독립당을 언급한 문서가 몇 편인지 세고 숫자로 답한다")
print(why, "|", answer)

완료 | 16


트레이스에는 턴마다 부른 도구, 누적 토큰, 경과 시간이 남는다.

In [9]:
for t in trace:
    print(f"  턴 {t['턴']}  {t['초']:4.1f}초  {t['토큰']:6,d} 토큰  {t['도구']}")

  턴 1   2.4초     176 토큰  ['grep_docs']
  턴 2   3.4초     617 토큰  []


### CONFIG만 바꿔서 행동을 바꾼다

코드는 한 줄도 건드리지 않는다. 이 표면이 Part 4 `17`의 개선 루프가 만지는 곳이다.
분리해 두지 않으면 그 회차에서 고칠 것이 프롬프트 문자열 하나밖에 없다.

In [ ]:
def with_config(**over):    ## 오버라이드 
    saved = dict(CONFIG)
    CONFIG.update(over)
    try:
        return run("한국독립당과 조선민족혁명당을 언급한 문서 수를 각각 센다")
    finally:
        CONFIG.clear()
        CONFIG.update(saved)

기본 설정으로 한 번 돌린다.

In [11]:
answer, why, trace = with_config()
print(f"{why:6} 턴 {len(trace)}  도구 {[t['도구'] for t in trace]}")
print(str(answer)[:80])

완료     턴 2  도구 [['grep_docs', 'grep_docs'], []]
한국독립당: 16문서  
조선민족혁명당: 8문서


반복 예산을 1로 줄이거나 지시문을 바꾸면 같은 코드가 다르게 움직인다.

In [12]:
for label, over in [("반복 예산 1", {"max_turns": 1}),
                    ("지시 바꿈", {"instructions": "도구를 쓰지 말고 아는 대로 답한다."})]:
    answer, why, trace = with_config(**over)
    print(f"{label:10} {why:6} 턴 {len(trace)}  도구 {[t['도구'] for t in trace]}")
    print(f"           {str(answer)[:56]}")

반복 예산 1    반복 예산  턴 1  도구 [['grep_docs', 'grep_docs']]
           None
지시 바꿈      완료     턴 1  도구 [[]]
           제가 가진 대화 내 정보만으로는 실제 문서 집합을 조회할 수 없어서 정확한 “문서 수”를 셀 수는 없


### Policy는 거부도 기록한다

`2`에서 승인 게이트를 인자로 넘겨 봤다. 코어에서는 그것이 `CONFIG["gated"]` 한 줄이다.
막은 것을 트레이스에 남기지 않으면 나중에 무엇이 막혔는지 알 수 없다.

In [13]:
answer, why, trace = run("한국독립당 문서 수를 세어 count.txt 에 저장한다", approve=False)
print("막힘 |", str(answer)[:80])
print([t["도구"] for t in trace])

막힘 | `한국독립당`이 나온 문서는 16개로 확인됐다.

`count.txt`에 `16`을 저장하려면 승인해 달라.
[['grep_docs'], ['write_note'], []]


승인하면 파일이 생긴다.

In [14]:
answer, why, trace = run("한국독립당 문서 수를 세어 count.txt 에 저장한다", approve=True)
print("승인 |", [f.name for f in Path("work").iterdir()])

승인 | ['count.txt']


In [15]:
shutil.rmtree("work")

### 코어 복제 규칙

이후 노트북은 코어를 import하지 않고 자기 안에 복제한다. 노트북 하나만 열어도 실행되어야 하고, 학생이 루프 전체를 그 노트북 안에서 읽을 수 있어야 하기 때문이다.

회차의 주제가 도구라면 `TOOLS`, `IMPL`, `CONFIG`만 바꾸고 코어는 그대로 쓴다. 주제가 하네스의 한 부분이면 그 부분을 코어 안에서 고친다. `6`은 ToolRegistry에 코드 실행을 붙이고, `7`은 `run`에 기억을 앞에 붙이는 인자를 더하며, `8`은 Trace를 스팬 트리로 바꾼다. Part 3, 4에서는 그 회차에 필요 없는 부분을 덜어 내거나 반환 형태를 실험에 맞게 바꾼다.

바뀌지 않는 것은 뼈대다. 모델을 부르고, 도구 호출을 모아 실행하고, 결과를 붙여 다시 부르다가 도구 호출이 없거나 반복 예산이 다하면 멈춘다. 

### 비용 확인

In [16]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 12회
입력 3,791 토큰   출력 477 토큰
누적 $0.0166
